# Exercise 1. Prepare Big Data for ML!

## 1.1 Introducing the Data

As mentioned, you will be working with {cite:t}`dugan-etal-2024-raid`'s RAID dataset. The files can be found in `resources/data/raid` on `UCloud` (should be mounted if you followed [Class Setup](class-setup)):
```bash
└── raid
    ├── test.csv
    ├── test_none.csv <-- NEVER EVALUATE ON TEST BEFORE BEING DONE WITH TRAIN!!
    ├── train.csv
    └── train_none.csv
```

You will be working with the `train_none.csv` which contains the following:
```{figure} ../figures/class2/raid_figure.png
---
name: raid-overview
---
Figure modified from {cite:t}`dugan-etal-2024-raid`.
```

`train_none.csv` is a subset of the entire dataset. The full dataset also contains `adversarial attacks` (6.2M examples in total!). For simplicity, we won't be looking at those today.

:::{admonition} What are adversarial attacks?
:class: dropdown, tip
Adversarial attacks are carefully designed modifications to input data (in our case, text) that can cause a classifier to make incorrect predictions. They exploit weaknesses in the model that were not encountered during training. Incorporating such attacks into the training process can improve the classifier’s robustness against unexpected or manipulated inputs in real-world settings.

In {cite:t}`dugan-etal-2024-raid`'s RAID, these include everything from British spelling, article deletions, and mispellings. Read more about it in their paper!

I have downloaded the full dataset `train.csv` for you to explore if you like. It contains 6.2M examples, so consider choosing a larger UCloud machine if it loads slowly.
:::

### Load the Data
Start by importing `pathlib` and `pandas`:

In [68]:
from pathlib import Path
import pandas as pd

Define paths:

In [69]:
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

In [70]:
raw_df = pd.read_csv(data_path)

Let's look at how many rows there is in our df:

In [71]:
print(len(raw_df))

467985


### Your Turn: Look at the raw data
```{admonition} HANDS-ON
:class: red

1. Load `raw_df` in your notebook if you haven't already! 
2. Print all column names  `raw_df` 
3. Do you notice any columns that you might not immediately know what corresponds to? Read up on [the column names](https://huggingface.co/datasets/liamdugan/raid#data-fields) before proceeding!
4. From {numref}`raid-overview`, we have gotten an overview of the kinds of LLMs used in this dataset, but what are they called in our dataframe? Find all unique values in the `models` column.
```

#### Print Column Names

```{admonition} HINT
:class: tip, dropdown
Look at the .columns attribute on Pandas - see [docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) for help
```

Check the solution:

In [72]:
columns_in_df = raw_df.columns.tolist() # you don't technically need .tolist() - it it just to get it in a neat list (try to remove to see effect) 
print(columns_in_df)

['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']


#### Unique Models

```{admonition} HINT
:class: tip, dropdown
What is the LLM column called? You need this, and then you can use the `.unique()` method:
https://pandas.pydata.org/docs/reference/api/pandas.unique.html
```

Solution below:

In [73]:
unique_models = raw_df["model"].unique()
print(unique_models)

['human' 'llama-chat' 'mpt' 'mpt-chat' 'gpt2' 'mistral' 'mistral-chat'
 'gpt3' 'cohere' 'chatgpt' 'gpt4' 'cohere-chat']


### Subset Data
For now, we want to look only at `human` and `chatgpt` generations! Let's use the `isin()` function that we also played with at the end of [Class 1 (Section 2.3)](23-parts-of-speech-analysis=)

In [74]:
df = raw_df[raw_df["model"].isin(["human", "chatgpt"])]

Let's see how many chatgpt and human generations with the `group.by` function, applying `size()` to it:

In [75]:
df.groupby("model").size()

model
chatgpt    26742
human      13371
dtype: int64

:::{admonition} QUESTION
:class: red

Do you know why it is a problem that we have double the amount of `chatgpt` generations?

Consider this with your group and click to reveal answer before proceeding.

```{dropdown} Click to see ANSWER
Classification models generally assume that all classes in a dataset have roughly the same number of examples. Unbalanced classes can cause the classifier to perform poorly on the under-represented class, also called the `minority class` (see {cite:t}`taskiran_comprehensive_2025`).
```
:::

## 1.2 Fixing Unbalanced Classes
There are many ways to approach having unbalanced classes. As seen on {numref}`raid-overview`, the reason we have more `chatgpt` rows is due to having different generation parameters such as `decoding` where two sets of `chatgpt` generations is created (one with `greedy` and with `sampling`). 

```{admonition} LLM FRAMING
:class: fuchsia
In brief, `greedy` decoding and `sampling` are methods that determine how an LLM selects words when generating text. You will learn more about them later in the course.
```

We could just filter away either `greedy` or `sampling` to balance the classes, but that would be booooring. Soooo.... let's make you do something else!

### Your Turn: "Downsampling" 
:::{admonition} HANDS-ON
:class: red

We want to "downsample" the majority class (i.e., remove data points), but keep both `greedy` and `sampling` represented !

**YOUR TASK** : Downsample the majority class `chatgpt`, keeping half of `greedy` and half of `sampling`
    1. Create why 

There will be hints below! 
:::

## 1.2 Create Training Splits!

We will start by 